# Solar Flux Prediction

In [ ]:
import torch

device_try = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Selected device: {device_try}")

In [ ]:
import os

print("=== TEST TMUX ===")
tmux_var = os.environ.get("TMUX")

if tmux_var:
    print("STATUS: Jupyter server is running in a TMUX session!")
    print(f"Tmux socket path: {tmux_var}")
else:
    print("STATUS: The Jupyter server is NOT inside Tmux (it runs in the global terminal).")

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
print("--- Caricamento e Aggregazione Giornaliera ---") # 🛠️ Modificato testo log
# Carichiamo il dataset originale (non quello con il lag precedente)
df = pd.read_csv("Data/MARSIS_historical_dataset.csv", sep=";")
df.columns = df.columns.str.strip()

# Convertiamo l'Ephemeris Time (secondi dal J2000) in oggetti Datetime reali
df['datetime'] = pd.to_datetime(df['FM_data_ephemeris_time'], unit='s', origin=pd.Timestamp('2000-01-01 12:00:00'))

# 🛠️ CHIRURGIA 1: Cambiato 'M' in 'D' per raggruppare per Giorno (Day) invece che per Mese
df_daily = df.groupby(df['datetime'].dt.to_period('D')).agg({
    'FM_data_F10_7_index': 'mean'
}).reset_index()

df_daily['datetime'] = df_daily['datetime'].dt.to_timestamp()

# 🛠️ CHIRURGIA 2: Sostituito/Arricchito con il giorno dell'anno (da 1 a 365)
# Questo dà alla LSTM un'informazione temporale molto più precisa per i dati giornalieri
df_daily['day_of_year'] = df_daily['datetime'].dt.dayofyear
df_daily['month_of_year'] = df_daily['datetime'].dt.month 

print(f"Dataset giornaliero creato. Totale giorni disponibili: {len(df_daily)}") # 🛠️ Modificato testo log

In [ ]:
# Setta la data come indice, riempie i giorni mancanti ripetendo l'ultimo valore (Forward Fill) e resetta l'indice
df_daily = df_daily.set_index('datetime').asfreq('D', method='ffill').reset_index()

In [ ]:
# Usiamo gli ultimi 30 giorni per predire il flusso solare di "domani"
LOOKBACK_WINDOW = 30 

# Creiamo le matrici usando il dataset giornaliero e il giorno dell'anno
X_flux, X_time, y = create_dataset_windows(
    data=df_daily, 
    target_col='FM_data_F10_7_index', 
    time_col='day_of_year',  # <--- Cruciale!
    lookback=LOOKBACK_WINDOW
)

# Split Cronologico (80% Train, 20% Test)
split_idx = int(len(y) * 0.8)

X_flux_train, X_flux_test = X_flux[:split_idx], X_flux[split_idx:]
X_time_train, X_time_test = X_time[:split_idx], X_time[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

## Linear regression for solar flux prediction 

In [ ]:
print("\n--- Training: Regressione Lineare ---")

# Uniamo le feature (Flusso_t-12...Flusso_t + Mese_t-12...Mese_t)
X_linear_train = np.hstack((X_flux_train, X_train_time_placeholder := X_time_train))
X_linear_test = np.hstack((X_flux_test, X_test_time_placeholder := X_time_test))

# Inizializzazione e fit
lr_model = LinearRegression()
lr_model.fit(X_linear_train, y_train)

# Predizione
y_pred_lr = lr_model.predict(X_linear_test)

# Valutazione
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print(f"[Linear Regression] Test MAE: {mae_lr:.3f}, Test RMSE: {rmse_lr:.3f}")